# Join of the dastasets

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from src.utils import data_understanding as du
from src.utils import utils as ut

hd_df = pd.read_csv("../../data/processed/hd.csv")
lab_events_df = pd.read_csv("../../data/processed/lab_events.csv")
micro_bio_df = pd.read_csv("../../data/processed/microbio.csv")
procedures_df = pd.read_csv("../../data/processed/procedure_code_summary.csv")

# Make the join between processed datasets

joined_df = pd.merge(hd_df, lab_events_df, on='hadm_id', how='left', suffixes=('_hd', '_le'))

joined_df = pd.merge(
    joined_df,
    micro_bio_df,
    on=['subject_id', 'hadm_id'],
    how='left',
    suffixes=('_joined', '_micro')
)

joined_df = pd.merge(
    joined_df,
    procedures_df,
    on=['subject_id', 'hadm_id'],
    how='left',
    suffixes=('_joined', '_proc')
)

# Drop ALL duplicate rows (keeps only unique rows)
joined_df = joined_df.drop_duplicates().reset_index(drop=True)
joined_df

In [ ]:
#Filling the missing values for those who did not have a lab event done
joined_df['valuenum'] = joined_df['valuenum'].fillna(value = -1)
joined_df['valueuom'] = joined_df['valueuom'].fillna(value = "NO_UOM")
joined_df['flag'] = joined_df['flag'].fillna(value = -1)
joined_df['label'] = joined_df['label'].fillna(value = "NO_LAB_LABEL")
joined_df['fluid'] = joined_df['fluid'].fillna(value = "NO_FLUID")
joined_df['examination_group'] = joined_df['examination_group'].fillna(value = "NO_EXAM_GROUP")
joined_df['analysis_batch_id'] = joined_df['analysis_batch_id'].fillna(value = "NO_BATCH_ID")

# Filling the missing values for those who did not have any procedure done
joined_df['number_unique_procedures'] = joined_df['number_unique_procedures'].fillna(value = 0)
joined_df['icd_code_proc'] = joined_df['icd_code_proc'].fillna(value = "NO_PROCEDURES")

joined_df

In [ ]:
# Filling the missing values for those who did not have a microbio test done
joined_df['spec_type_desc'] = joined_df['spec_type_desc'].fillna(value = "NO_MICROBIO")
joined_df

#If NO_MICROBIO, fill the microbio_test fields 
micro_bio_cols = ['org_name', 'ab_name', 'dilution_text','dilution_comparison','dilution_value', 'interpretation', 'qc_flag']

micro_bio_mask = joined_df['spec_type_desc'] == "NO_MICROBIO"

#Fill orgname with NOT_ORG_DETECTED
joined_df.loc[micro_bio_mask, 'org_name'] = joined_df.loc[micro_bio_mask, 'org_name'].fillna(value = "NO_ORG_DETECTED")

#Fill test_name with NO_TEST_DONE
joined_df.loc[micro_bio_mask, 'test_name'] = joined_df.loc[micro_bio_mask, 'ab_name'].fillna(value = "NO_TEST_DONE")

# Fill ab_name with NO_AB_DETECTED
joined_df.loc[micro_bio_mask, 'ab_name'] = joined_df.loc[micro_bio_mask, 'ab_name'].fillna(value = "NO_AB_DETECTED")

#Fill dilution_text with "-1"
joined_df.loc[micro_bio_mask, 'dilution_text'] = joined_df.loc[micro_bio_mask, 'dilution_text'].fillna(value = "-1")

#Fill dilution_comparison with "="
joined_df.loc[micro_bio_mask, 'dilution_comparison'] = joined_df.loc[micro_bio_mask, 'dilution_comparison'].fillna(value = "=")

#Fill dilution_value with -1
joined_df.loc[micro_bio_mask, 'dilution_value'] = joined_df.loc[micro_bio_mask, 'dilution_value'].fillna(value = -1)

#Fill interpretation with NOT_INTERPRETATION
joined_df.loc[micro_bio_mask, 'interpretation'] = joined_df.loc[micro_bio_mask, 'interpretation'].fillna(value = "NO_INTERPRETATION")

#Fill qc_flag with NOT_QC_FLAG
joined_df.loc[micro_bio_mask, 'qc_flag'] = joined_df.loc[micro_bio_mask, 'qc_flag'].fillna(value = "NO_QC_FLAG")

joined_df


## Correlation between the cols

In [ ]:
# Getting the correlations
numeric_cols = du.get_numeric_cols(joined_df)
# Pearson correlation
joined_df[numeric_cols].corr("pearson")

In [ ]:
# Kendall correlation
joined_df[numeric_cols].corr("kendall")

In [ ]:
# Spearman correlation
joined_df[numeric_cols].corr("spearman")

# Feature creation

In [ ]:
# Creating the new features representing the profiles of the patients
#Taking the common attributes (subject_id,hadm_id, gender, age, ICD)
profile_cols = ['subject_id', 'hadm_id', 'gender', 'age', 'icd_code_joined']
profile_df = joined_df[profile_cols].drop_duplicates().reset_index(drop=True)

# === LAB PROFILE FEATURES === #

#Creating the feature for the lab events, if "NO_LAB_LABEL" then count is 0
lab_event_counts = joined_df.groupby(['subject_id', 'hadm_id'])['label'].apply(lambda x: (x != "NO_LAB_LABEL").sum()).reset_index()
lab_event_counts = lab_event_counts.rename(columns={'label': 'lab_event_count'})
profile_df = pd.merge(profile_df, lab_event_counts, on=['subject_id', 'hadm_id'], how='left')

# Creating the ratio of abnormal lab events to total lab events, if no lab events then ratio is 0
abnormal_lab_event_counts = joined_df[joined_df['flag'] == 1].groupby(['subject_id', 'hadm_id'])['flag'].count().reset_index()
abnormal_lab_event_counts = abnormal_lab_event_counts.rename(columns={'flag': 'abnormal_lab_event_count'})
profile_df = pd.merge(profile_df, abnormal_lab_event_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['abnormal_lab_event_count'] = profile_df['abnormal_lab_event_count'].fillna(0)

profile_df['abnormal_lab_event_ratio'] = profile_df.apply(
    lambda row: row['abnormal_lab_event_count'] / row['lab_event_count'] if row['lab_event_count'] > 0 else 0,
    axis=1
)

# Creating the lab events density using the charttime
lab_event_times = joined_df[joined_df['label'] != "NO_LAB_LABEL"].groupby(['subject_id', 'hadm_id'])['charttime_joined'].agg(['min', 'max']).reset_index()
lab_event_times['lab_event_duration_days'] = (pd.to_datetime(lab_event_times['max']) - pd.to_datetime(lab_event_times['min'])).dt.days + 1
profile_df = pd.merge(profile_df, lab_event_times[['subject_id', 'hadm_id', 'lab_event_duration_days']], on=['subject_id', 'hadm_id'], how='left')
profile_df['lab_event_duration_days'] = profile_df['lab_event_duration_days'].fillna(1)
profile_df['lab_event_density'] = profile_df.apply(
    lambda row: row['lab_event_count'] / row['lab_event_duration_days'] if row['lab_event_duration_days'] > 0 else 0,
    axis=1
)

# === MICROBIO PROFILE FEATURES === #

#Counting the number of microbio tests done per subject_id and hadm_id, if "NO_MICROBIO" then count is 0
microbio_test_counts = joined_df[joined_df['spec_type_desc'] != "NO_MICROBIO"].groupby(['subject_id', 'hadm_id'])['spec_type_desc'].nunique().reset_index()
microbio_test_counts = microbio_test_counts.rename(columns={'spec_type_desc': 'microbio_test_count'})
profile_df = pd.merge(profile_df, microbio_test_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['microbio_test_count'] = profile_df['microbio_test_count'].fillna(0)

# Counting the number of unique microorganisms detected per subject_id and hadm_id, if "NO_ORG_DETECTED" then count is 0
microbio_counts = joined_df[joined_df['org_name'] != "NO_ORG_DETECTED"].groupby(['subject_id', 'hadm_id'])['org_name'].nunique().reset_index()
microbio_counts = microbio_counts.rename(columns={'org_name': 'unique_microorganisms_count'})
profile_df = pd.merge(profile_df, microbio_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['unique_microorganisms_count'] = profile_df['unique_microorganisms_count'].fillna(0)

# Ratio of resistant microorganisms to total microorganisms detected, if intepretation is "NO_INTERPRETATION" then count is 0
resistant_microbio_counts = (
    joined_df[(joined_df["interpretation"] == "R") & (joined_df["org_name"] != "NO_ORG_DETECTED")]
    .drop_duplicates(subset=["subject_id", "hadm_id", "org_name"])
    .groupby(["subject_id", "hadm_id"])["org_name"]
    .nunique()
    .reset_index(name="resistant_microorganisms_count")
)
profile_df = pd.merge(profile_df, resistant_microbio_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['resistant_microorganisms_count'] = profile_df['resistant_microorganisms_count'].fillna(0)
profile_df['resistant_microorganism_ratio'] = profile_df.apply(
    lambda row: row['resistant_microorganisms_count'] / row['unique_microorganisms_count'] if row['unique_microorganisms_count'] > 0 else 0,
    axis=1
)

# Ratio of intermediate microorganisms to total microorganisms detected, if intepretation is "NO_INTERPRETATION" then count is 0
intermediate_microbio_counts = (
    joined_df[(joined_df["interpretation"] == "I") & (joined_df["org_name"] != "NO_ORG_DETECTED")]
    .drop_duplicates(subset=["subject_id", "hadm_id", "org_name"])
    .groupby(["subject_id", "hadm_id"])["org_name"]
    .nunique()
    .reset_index(name="intermediate_microorganisms_count")
)

profile_df = pd.merge(profile_df, intermediate_microbio_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['intermediate_microorganisms_count'] = profile_df['intermediate_microorganisms_count'].fillna(0)
profile_df['intermediate_microorganism_ratio'] = profile_df.apply(
    lambda row: row['intermediate_microorganisms_count'] / row['unique_microorganisms_count'] if row['unique_microorganisms_count'] > 0 else 0,
    axis=1
)

# Ratio of sensitive microorganisms to total microorganisms detected, if intepretation is "NO_INTERPRETATION" then count is 0
sensitive_microbio_counts = (
    joined_df[(joined_df["interpretation"] == "S") & (joined_df["org_name"] != "NO_ORG_DETECTED")]
    .drop_duplicates(subset=["subject_id", "hadm_id", "org_name"])
    .groupby(["subject_id", "hadm_id"])["org_name"]
    .nunique()
    .reset_index(name="sensitive_microorganisms_count")
)
profile_df = pd.merge(profile_df, sensitive_microbio_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['sensitive_microorganisms_count'] = profile_df['sensitive_microorganisms_count'].fillna(0)
profile_df['sensitive_microorganism_ratio'] = profile_df.apply(
    lambda row: row['sensitive_microorganisms_count'] / row['unique_microorganisms_count'] if row['unique_microorganisms_count'] > 0 else 0,
    axis=1
)

# Creating the microbio test density using the charttime_microbio
microbio_test_times = joined_df[joined_df['spec_type_desc'] != "NO_MICROBIO"].groupby(['subject_id', 'hadm_id'])['charttime_micro'].agg(['min', 'max']).reset_index()
microbio_test_times['microbio_test_duration_days'] = (pd.to_datetime(microbio_test_times['max']) - pd.to_datetime(microbio_test_times['min'])).dt.days + 1
profile_df = pd.merge(profile_df, microbio_test_times[['subject_id', 'hadm_id', 'microbio_test_duration_days']], on=['subject_id', 'hadm_id'], how='left')
profile_df['microbio_test_duration_days'] = profile_df['microbio_test_duration_days'].fillna(1)
profile_df['microbio_test_density'] = profile_df.apply(
    lambda row: row['microbio_test_count'] / row['microbio_test_duration_days'] if row['microbio_test_duration_days'] > 0 else 0,
    axis=1
)

# === PROCEDURE PROFILE FEATURES === #
# Counting the number of unique procedures done per subject_id and hadm_id, if "NO_PROCEDURES" then count is 0
procedure_counts = joined_df[joined_df['icd_code_proc'] != "NO_PROCEDURES"].groupby(['subject_id', 'hadm_id'])['icd_code_proc'].nunique().reset_index()
procedure_counts = procedure_counts.rename(columns={'icd_code_proc': 'unique_procedures_count'})
profile_df = pd.merge(profile_df, procedure_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['unique_procedures_count'] = profile_df['unique_procedures_count'].fillna(0)

# Taking the total procedures codes per subject_id and hadm_id, if "NO_PROCEDURES" then "NO PROCEDURES"
total_procedure_counts = joined_df.groupby(['subject_id', 'hadm_id'])['number_unique_procedures'].max().reset_index()
total_procedure_counts = total_procedure_counts.rename(columns={'number_unique_procedures': 'total_procedures_count'})
profile_df = pd.merge(profile_df, total_procedure_counts, on=['subject_id', 'hadm_id'], how='left')
profile_df['total_procedures_count'] = profile_df['total_procedures_count'].fillna(0)

# Removes all the counts columns used for calculations
cols = profile_df.columns
counts_cols = [col for col in cols if 'count' in col and col not in ['subject_id', 'hadm_id']]
profile_df = profile_df.drop(columns=counts_cols)

# Remove duplicates (just in case)
profile_df = profile_df.drop_duplicates().reset_index(drop=True)


profile_df

## Visual exploration
The following plots give a quick overview of age distribution in the cohort, the tests with the highest abnormal rates, and how resistance relates to antibiotic coverage.

In [ ]:
# Age distribution with KDE overlay and highlighted quartiles (unique subjects)
unique_subject_age = (
    joined_df[['subject_id', 'age']]
    .dropna(subset=['age'])
    .drop_duplicates(subset='subject_id')
)

plt.figure(figsize=(10, 4))
sns.histplot(
    data=unique_subject_age,
    x='age',
    bins=30,
    kde=True,
    color='#3A7CA5',
    edgecolor='white'
)
q1, q2, q3 = unique_subject_age['age'].quantile([0.25, 0.5, 0.75])
for q, label in zip([q1, q2, q3], ['Q1', 'Median', 'Q3']):
    plt.axvline(q, color='#E57F84', linestyle='--', linewidth=1, label=label if label == 'Median' else None)
plt.title('Age distribution of the joined cohort (unique subjects)')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(loc='upper right')
plt.tight_layout()

## Sex distribution per ICD code

In [ ]:
counts = (
    joined_df.groupby(['icd_code_joined', 'gender'])['subject_id']
    .nunique()
    .reset_index(name='unique_patients')
)

icd_order = (
    counts.groupby('icd_code_joined')['unique_patients']
    .sum()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(10, 4))
sns.barplot(
    data=counts,
    x='icd_code_joined',
    y='unique_patients',
    hue='gender',
    palette='Set2',
    order=icd_order
)
plt.title('Unique patients per ICD code by sex')
plt.xlabel('ICD code')
plt.ylabel('Unique patients')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
subject_age = (
    joined_df[['subject_id', 'gender', 'age']]
    .dropna(subset=['gender', 'age'])
    .drop_duplicates(subset='subject_id')
)

plt.figure(figsize=(10, 8), dpi=300)
sns.boxplot(
    data=subject_age,
    x='gender',
    y='age',
    hue='gender',
    palette='Set2',
    width=0.6,
    dodge=False,
    legend=False
)
plt.title('Age distribution by gender (unique subjects)')
plt.xlabel('Gender')
plt.ylabel('Age')
plt.tight_layout()

In [ ]:
#to visualize for each icd_code the age of the patient 
icd_age = (
    joined_df[['subject_id', 'icd_code_joined', 'age']]
    .dropna(subset=['icd_code_joined', 'age'])
    .drop_duplicates(subset=['subject_id', 'icd_code_joined'])
)

plt.figure(figsize=(10, 8), dpi=300)
sns.boxplot(
    data=icd_age,
    x='icd_code_joined',
    y='age',
    color='#3A7CA5'
)
plt.title('Age distribution per ICD code (unique subjects)')
plt.xlabel('ICD code')
plt.ylabel('Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
icd_order = (
    icd_age.groupby('icd_code_joined')['subject_id']
    .nunique()
    .sort_values(ascending=False)
    .index
)

median_age_by_icd = (
    icd_age.groupby('icd_code_joined')['age']
    .median()
    .reindex(icd_order)
)

plt.figure(figsize=(10, 8), dpi=300)
sns.stripplot(
    data=icd_age,
    x='icd_code_joined',
    y='age',
    order=icd_order,
    color='#E57F84',
    jitter=0.25,
    size=3,
    alpha=0.6
)

for idx, age in enumerate(median_age_by_icd):
    plt.hlines(age, idx - 0.3, idx + 0.3, color='#3A7CA5', linewidth=1.2)

plt.title('Age vs ICD code (unique subjects)')
plt.xlabel('ICD code')
plt.ylabel('Age')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
# Prima conta i valori della colonna flag
counts = joined_df['flag'].value_counts().sort_index()

# Poi crea il grafico a torta
plt.figure(figsize=(10, 8), dpi=300)
plt.pie(counts.values, 
        labels=counts.index,  # oppure ['Normale', 'Anormale']
        autopct='%1.1f%%',
        colors=['lightgreen', 'salmon'])
plt.title('Distribuzione valori flag')
plt.show()

In [ ]:
org_count = np.log10(joined_df['org_name'].value_counts().replace(0, np.nan))

plt.figure(figsize=(20, 12), dpi=300)
sns.barplot(y=org_count.index, x=org_count.values)
plt.title("Organism counts distribution (log10 scaled)", fontsize=20, fontweight="bold")
plt.ylabel("Organism name")
plt.xlabel("counts")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
test_count = np.log10(joined_df['test_name'].value_counts().replace(0, np.nan))

plt.figure(figsize=(20, 12))
sns.barplot(y=test_count.index, x=test_count.values)
plt.title("Test counts distribution (log10 scaled)", fontsize=20, fontweight="bold")
plt.ylabel("Test name")
plt.xlabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
ab_count =joined_df['ab_name'].value_counts()

plt.figure(figsize=(20, 12), dpi=300)
sns.barplot(x=ab_count.index, y=ab_count.values)
plt.title("Antibiotic distribution plot", fontsize=20, fontweight="bold")
plt.xlabel("Antibiotic name")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Get the counts
resistant_counts = np.log10(joined_df.groupby("ab_name")['interpretation'].value_counts().replace(0, np.nan))
counts_df = resistant_counts.reset_index(name='count')

# Calculate average count for each interpretation to determine order
interpretation_avg = counts_df.groupby('interpretation')['count'].mean().sort_values(ascending=True)
sorted_interpretations = interpretation_avg.index.tolist()

# Create the plot
fig, ax = plt.subplots(figsize=(20,12), dpi=300)

# Create grouped barplot with sorted hue order
sns.barplot(
    data=counts_df,
    x='ab_name',
    y='count',
    hue='interpretation',
    ax=ax,
    palette='Set2',
    hue_order=sorted_interpretations
)

# Customize the plot
ax.set_title('Antibiotic Resistance Interpretation Counts', fontsize=20, fontweight='bold')
ax.set_xlabel('Antibiotic Name', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(title='Interpretation', title_fontsize=11, fontsize=10)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
# Get unique interpretations
interpretations = counts_df['interpretation'].unique()
n_interpretations = len(interpretations)

# Create subplots - one for each interpretation
fig, axes = plt.subplots(n_interpretations, 1, figsize=(10,  6* n_interpretations), dpi=300)

# Handle case of single interpretation
if n_interpretations == 1:
    axes = [axes]

# Create a plot for each interpretation
for idx, interpretation in enumerate(interpretations):
    # Filter data for this interpretation
    subset = counts_df[counts_df['interpretation'] == interpretation]
    
    # Sort by count in descending order
    subset = subset.sort_values('count', ascending=False)
    
    # Create barplot
    sns.barplot(
        data=subset,
        x='ab_name',
        y='count',
        ax=axes[idx],
        color=sns.color_palette('Set2')[idx % 10],
        order=subset['ab_name']  # Explicitly set the order
    )
    
    # Customize the subplot
    axes[idx].set_title(f'{interpretation} - Counts by Antibiotic (log10 scaled)', 
                        fontsize=20, fontweight='bold')
    axes[idx].set_xlabel('Antibiotic Name', fontsize=12)
    axes[idx].set_ylabel('Count', fontsize=12)
    
    # Rotate x-axis labels
    axes[idx].tick_params(axis='x', rotation=45)
    plt.setp(axes[idx].xaxis.get_majorticklabels(), ha='right')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()